# Tutorial 3: S3 with NumPy and PyTorch Tensors

Store and recall numerical data on AWS S3. The workflow is the same as local pools, but the pool talks to a remote bucket.

**Prerequisites:** `pip install "laila-core[s3,torch]"` and a `secrets.toml` with your AWS credentials.

In [ ]:
import numpy as np
import torch
import laila
from laila.pool import S3Pool

## Load credentials

Create a `secrets.toml` in the same directory:
```toml
AWS_BUCKET_NAME = "your-bucket"
AWS_ACCESS_KEY_ID = "AKIA..."
AWS_SECRET_ACCESS_KEY = "wJa..."
AWS_REGION = "us-east-1"
```

In [ ]:
laila.read_args("./secrets.toml")

## Create and register the S3 pool

In [ ]:
s3_pool = S3Pool(
    bucket_name=laila.args.AWS_BUCKET_NAME,
    access_key_id=laila.args.AWS_ACCESS_KEY_ID,
    secret_access_key=laila.args.AWS_SECRET_ACCESS_KEY,
    region_name=laila.args.AWS_REGION,
    nickname="s3",
)

laila.memory.extend(s3_pool, pool_nickname="s3")

## Store a numpy array

In [ ]:
matrix = np.random.randn(100, 100)
np_entry = laila.constant(data=matrix, nickname="np_matrix")

with laila.guarantee:
    laila.memorize(np_entry, pool_nickname="s3")

print(f"Stored global_id: {np_entry.global_id}")

## Store a PyTorch tensor

In [ ]:
image_tensor = torch.randn(3, 224, 224)
torch_entry = laila.constant(data=image_tensor, nickname="torch_img")

with laila.guarantee:
    laila.memorize(torch_entry, pool_nickname="s3")

print(f"Stored global_id: {torch_entry.global_id}")

## Recall from S3

Clear local references and recall purely from the stored `global_id`.

In [ ]:
np_gid = np_entry.global_id
torch_gid = torch_entry.global_id

del matrix, np_entry, image_tensor, torch_entry

In [ ]:
with laila.guarantee:
    np_future = laila.remember(np_gid, pool_nickname="s3")
recalled_np = np_future.data

print(f"NumPy shape: {recalled_np.shape}")
print(f"NumPy dtype: {recalled_np.dtype}")

In [ ]:
with laila.guarantee:
    torch_future = laila.remember(torch_gid, pool_nickname="s3")
recalled_torch = torch_future.data

print(f"Torch shape: {recalled_torch.shape}")
print(f"Torch dtype: {recalled_torch.dtype}")

## Inspecting futures

Every future is registered in the active policy's **future bank**. The
`laila.runtime` module is the canonical way to inspect them — it works
across local and remote (peer-owned) futures alike.

In [ ]:
print("Status:    ", laila.runtime.status(np_future))
print("Exception: ", laila.runtime.exception(np_future))

## Clean up

In [ ]:
with laila.guarantee:
    laila.forget(np_gid, pool_nickname="s3")
    laila.forget(torch_gid, pool_nickname="s3")
print("Cleaned up")

## Summary

- `S3Pool` takes a bucket name and AWS credentials. After `laila.memory.extend(...)` the API is identical to any local pool.
- numpy arrays and PyTorch tensors are serialized and deserialized automatically — types and shapes are preserved.
- Wrap I/O in `with laila.guarantee:` to block until every future created inside the block has completed.
- `laila.runtime.status(future)` and `laila.runtime.exception(future)` are the canonical introspection helpers and resolve futures by reference, identity, or `global_id` string.

**Next:** Tutorial 4 — Model Checkpoint and Reload